# 01 · Data overview

A first look at the ingested data: what we have, how predictable races are, and how the 2026
regulation reset has shuffled the field. Run `f1p ingest` first.

In [1]:
import pandas as pd
from scipy.stats import spearmanr
from sqlalchemy import text

from f1p.data.db import get_engine

pd.set_option("display.width", 120)
engine = get_engine()

races = pd.read_sql("select * from races", engine, parse_dates=["race_date"])
results = pd.read_sql(
    """select r.*, ra.season, ra.round, ra.circuit_id, ra.name as race_name
       from results r join races ra on ra.id = r.race_id where r.session = 'R'""",
    engine,
)
# SQLite stores booleans as 0/1.
results["classified"] = results.classified.astype(bool)
results.groupby("season").agg(
    races=("race_id", "nunique"), rows=("driver_id", "size"), drivers=("driver_id", "nunique")
)

,races,rows,drivers
season,,,
2014,19,407,24
2015,19,378,22
2016,21,462,24
2017,20,400,25
2018,21,420,20
2019,21,420,20
2020,17,340,23
2021,22,440,21
2022,22,440,22


## How often do races go to plan?

Share of races won from pole, and the Spearman correlation between grid and finishing order
(classified finishers only). This is the bar the post-qualifying model has to clear.

In [2]:
fin = results[results.classified & (results.grid > 0)]

def grid_corr(g):
    return spearmanr(g.grid, g.position).statistic if len(g) > 2 else float("nan")

per_race = fin.groupby(["season", "race_id"]).apply(grid_corr, include_groups=False).rename("rho")
pole_wins = results[results.position == 1].assign(from_pole=lambda d: d.grid == 1)
pd.DataFrame({
    "grid_finish_rho": per_race.groupby("season").mean(),
    "pole_win_rate": pole_wins.groupby("season").from_pole.mean(),
}).round(2)

,grid_finish_rho,pole_win_rate
season,,
2014,0.75,0.47
2015,0.78,0.63
2016,0.80,0.62
2017,0.77,0.55
2018,0.80,0.48
2019,0.74,0.38
2020,0.74,0.59
2021,0.75,0.50
2022,0.71,0.45


## Retirements

DNFs are a big source of noise. `classified` is false for retirements, disqualifications and
non-starters.

In [3]:
results.assign(dnf=~results.classified).groupby("season").dnf.mean().round(3).to_frame("dnf_rate")

,dnf_rate
season,
2014,0.197
2015,0.198
2016,0.171
2017,0.215
2018,0.190
2019,0.119
2020,0.156
2021,0.118
2022,0.145


## Overtaking difficulty by circuit

Mean grid→finish correlation per circuit, 2018 onward. High ρ = processional (qualifying decides
the race); low ρ = lots of position changes. This becomes a circuit feature.

In [4]:
circ = (
    per_race.reset_index()
    .merge(races[["id", "circuit_id"]], left_on="race_id", right_on="id")
    .query("season >= 2018")
    .groupby("circuit_id").rho.agg(["mean", "count"])
    .query("count >= 3").sort_values("mean")
)
pd.concat([circ.head(6), circ.tail(6)]).round(2)

,mean,count
circuit_id,,
vegas,0.49,3
interlagos,0.63,7
americas,0.67,7
sochi,0.67,4
zandvoort,0.69,6
monza,0.70,9
jeddah,0.82,5
miami,0.82,5
shanghai,0.83,5


## The 2026 reset: team pecking order

Average finishing position (by `position_order`, so DNFs count as last) per constructor, 2025 vs
2026. Big shifts show why recency weighting matters.

In [5]:
order = (
    results.query("season >= 2025")
    .groupby(["constructor_id", "season"]).position_order.mean()
    .unstack().round(1).sort_values(2026)
)
order

season,2025,2026
constructor_id,,
mercedes,7.2,5.1
ferrari,7.6,5.9
red_bull,8.1,8.0
mclaren,4.2,8.4
rb,12.3,10.2
alpine,15.4,11.3
audi,NaN,12.9
haas,12.1,13.8
williams,11.6,15.7


## Lap data sample

Practice long-run pace is the core pre-qualifying signal. One 2026 FP session: median clean lap
per driver, as % gap to the fastest.

In [6]:
from f1p import config

sessions = pd.read_sql(
    "select * from sessions where status = 'ok' order by race_id, session", engine
)
print(sessions.groupby(sessions.race_id.str[:4]).session.count().rename("sessions ok"))

fp = sessions[sessions.session.str.startswith("FP")].iloc[-1]
laps = pd.read_parquet(config.DATA_DIR / fp.laps_path)
clean = laps[laps.IsAccurate & ~laps.Deleted & laps.PitInTimeS.isna() & laps.PitOutTimeS.isna()
             & (laps.TrackStatus == "1")]
pace = clean.groupby("Driver").LapTimeS.median().sort_values()
print(fp.race_id, fp.session)
((pace / pace.min() - 1) * 100).round(2).rename("gap_%").to_frame().head(12)

race_id
2026    34
Name: sessions ok, dtype: int64
2026-7 FP3


,gap_%
Driver,
RUS,0.00
LEC,0.72
PIA,0.92
VER,1.03
BOR,1.73
LIN,1.75
LAW,1.90
HAD,2.04
NOR,2.35
